In [2]:
# ============================================================
# Cell 1 — Setup
# ============================================================
import requests
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, timedelta

os.makedirs("/kaggle/working/raw", exist_ok=True)
print("Setup complete.")

Setup complete.


In [4]:
## ============================================================
# Cell 2 — Secrets
# ============================================================
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

FIRMS_MAP_KEY = secrets.get_secret("FIRMS_MAP_KEY")
OPENAQ_API_KEY = secrets.get_secret("OPENAQ_API_KEY")
HF_TOKEN = secrets.get_secret("200205")

OPENAQ_HEADERS = {"X-API-Key": OPENAQ_API_KEY}
print("Secrets loaded.")

Secrets loaded.


In [6]:
# ============================================================
# Cell 3 — Region/season config (extended to 4 seasons)
# ============================================================
BBOX = "-124,36,-119,42"
BBOX_TUPLE = (-124, 36, -119, 42)

SEASONS = [
    {"start": "2019-08-01", "end": "2019-10-31"},
    {"start": "2020-08-01", "end": "2020-10-31"},
    {"start": "2021-08-01", "end": "2021-10-31"},
    {"start": "2022-08-01", "end": "2022-10-31"},
    {"start": "2023-08-01", "end": "2023-10-31"},
    {"start": "2024-08-01", "end": "2024-10-31"},
]
print(f"{len(SEASONS)} seasons configured: {[s['start'][:4] for s in SEASONS]}")

6 seasons configured: ['2019', '2020', '2021', '2022', '2023', '2024']


In [7]:
# ============================================================
# Cell 4 — FIRMS data availability check (verify before pulling)
# ============================================================
avail_url = f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{FIRMS_MAP_KEY}/VIIRS_SNPP_SP"
avail_df = pd.read_csv(avail_url)
print(avail_df)
# Confirm all four season date ranges fall within min_date/max_date before proceeding

         data_id    min_date    max_date
0  VIIRS_SNPP_SP  2012-01-20  2026-04-27


In [8]:
# ============================================================
# Cell 5 — FIRMS pull (VIIRS_SNPP_SP, day_range=5 — confirmed API constraint)
# ============================================================
def pull_firms(bbox, start_date, end_date, map_key, source="VIIRS_SNPP_SP", day_range=5):
    all_rows = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        date_str = current.strftime("%Y-%m-%d")
        url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{source}/{bbox}/{day_range}/{date_str}"
        try:
            df = pd.read_csv(url)
            if len(df) > 0:
                all_rows.append(df)
            print(f"  {date_str}: {len(df)} detections")
        except Exception as e:
            print(f"  {date_str}: FAILED - {e}")
        current += timedelta(days=day_range)
        time.sleep(1)

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

firms_frames = []
for season in SEASONS:
    print(f"Pulling FIRMS {season['start']} to {season['end']}")
    df = pull_firms(BBOX, season["start"], season["end"], FIRMS_MAP_KEY, source="VIIRS_SNPP_SP", day_range=5)
    firms_frames.append(df)

firms_df = pd.concat(firms_frames, ignore_index=True)
firms_df.to_csv("/kaggle/working/raw/firms_raw.csv", index=False)
print(f"\nTotal FIRMS detections: {len(firms_df)}")
firms_df.head()

Pulling FIRMS 2019-08-01 to 2019-10-31
  2019-08-01: 86 detections
  2019-08-06: 116 detections
  2019-08-11: 93 detections
  2019-08-16: 83 detections
  2019-08-21: 131 detections
  2019-08-26: 124 detections
  2019-08-31: 129 detections
  2019-09-05: 2998 detections
  2019-09-10: 1104 detections
  2019-09-15: 486 detections
  2019-09-20: 169 detections
  2019-09-25: 359 detections
  2019-09-30: 263 detections
  2019-10-05: 639 detections
  2019-10-10: 418 detections
  2019-10-15: 374 detections
  2019-10-20: 742 detections
  2019-10-25: 1644 detections
  2019-10-30: 230 detections
Pulling FIRMS 2020-08-01 to 2020-10-31
  2020-08-01: 543 detections
  2020-08-06: 365 detections
  2020-08-11: 929 detections
  2020-08-16: 22769 detections
  2020-08-21: 16838 detections
  2020-08-26: 8581 detections
  2020-08-31: 11490 detections
  2020-09-05: 37518 detections
  2020-09-10: 24219 detections
  2020-09-15: 13976 detections
  2020-09-20: 8784 detections
  2020-09-25: 13224 detections
  2020-

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,39.61945,-119.26413,309.04,0.78,0.78,2019-08-01,840,N,VIIRS,n,2,289.33,1.76,N,2
1,39.61947,-119.26048,302.06,0.78,0.78,2019-08-01,840,N,VIIRS,n,2,288.17,0.85,N,2
2,39.61949,-119.26202,318.31,0.43,0.46,2019-08-01,1020,N,VIIRS,n,2,291.01,2.68,N,2
3,40.73687,-122.32327,304.37,0.46,0.39,2019-08-01,1020,N,VIIRS,n,2,292.14,0.29,N,0
4,36.01361,-121.46650,321.72,0.40,0.44,2019-08-01,1021,N,VIIRS,n,2,291.26,2.24,N,0


In [9]:
# ============================================================
# Cell 6 — Filter to vegetation fires, high/nominal confidence
# ============================================================
firms_clean = firms_df[
    (firms_df["type"] == 0) &
    (firms_df["confidence"].isin(["n", "h"]))
].copy()

firms_clean["acq_datetime"] = pd.to_datetime(
    firms_clean["acq_date"] + " " + firms_clean["acq_time"].astype(str).str.zfill(4),
    format="%Y-%m-%d %H%M"
)

print(f"After filtering: {len(firms_clean)} detections (from {len(firms_df)} raw)")

After filtering: 349567 detections (from 368105 raw)


In [10]:
# ============================================================
# Cell 7 — Spatial clustering (DBSCAN)
# ============================================================
from sklearn.cluster import DBSCAN

coords = firms_clean[["latitude", "longitude"]].values
clustering = DBSCAN(eps=0.05, min_samples=3).fit(coords)
firms_clean["spatial_cluster"] = clustering.labels_

n_clusters = len(set(clustering.labels_)) - (1 if -1 in clustering.labels_ else 0)
print(f"Found {n_clusters} spatial fire clusters (excluding noise)")

Found 355 spatial fire clusters (excluding noise)


In [11]:
# ============================================================
# Cell 8 — Temporal splitting within spatial clusters (7-day gap = new event)
# ============================================================
GAP_THRESHOLD_DAYS = 7

event_rows = []
event_id = 0

for cluster_id, group in firms_clean[firms_clean["spatial_cluster"] != -1].groupby("spatial_cluster"):
    group = group.sort_values("acq_datetime")
    gaps = group["acq_datetime"].diff().dt.total_seconds() / 86400
    sub_event = (gaps > GAP_THRESHOLD_DAYS).cumsum()
    group = group.assign(sub_event=sub_event)

    for sub_id, sub_group in group.groupby("sub_event"):
        event_rows.append({
            "event_id": event_id,
            "spatial_cluster": cluster_id,
            "ignition_time": sub_group["acq_datetime"].min(),
            "last_detection_time": sub_group["acq_datetime"].max(),
            "lat": sub_group["latitude"].mean(),
            "lon": sub_group["longitude"].mean(),
            "n_detections": len(sub_group),
            "duration_days": (sub_group["acq_datetime"].max() - sub_group["acq_datetime"].min()).total_seconds() / 86400
        })
        event_id += 1

fire_events = pd.DataFrame(event_rows)
print(f"Distinct fire events after temporal split: {len(fire_events)}")

Distinct fire events after temporal split: 1355


In [12]:
# ============================================================
# Cell 9 — Quality filter (drop unreliable single/few-detection blips)
# ============================================================
MIN_DETECTIONS = 3
MIN_DURATION_DAYS = 0.5

fire_events_final = fire_events[
    (fire_events["n_detections"] >= MIN_DETECTIONS) &
    (fire_events["duration_days"] >= MIN_DURATION_DAYS)
].copy()

fire_events_final["season"] = fire_events_final["ignition_time"].dt.year

print(f"Final fire events: {len(fire_events_final)}")
print(fire_events_final["season"].value_counts().sort_index())
fire_events_final.to_csv("/kaggle/working/raw/fire_events_final.csv", index=False)
fire_events_final.head()

Final fire events: 452
season
2019     93
2020     65
2021     63
2022     59
2023    101
2024     71
Name: count, dtype: int64


,event_id,spatial_cluster,ignition_time,last_detection_time,lat,lon,n_detections,duration_days,season
0,0,0,2019-08-01 10:20:00,2019-08-13 09:55:00,40.736805,-122.322745,4,11.982639,2019
1,1,0,2019-08-22 20:09:00,2019-08-23 10:08:00,40.701815,-122.263765,28,0.582639,2019
7,7,0,2020-08-14 10:14:00,2020-08-30 10:14:00,40.736580,-122.324407,6,16.000000,2020
9,9,0,2021-08-15 20:34:00,2021-08-17 10:14:00,40.702193,-122.348710,3,1.569444,2021
11,11,0,2021-09-23 10:21:00,2021-09-26 20:47:00,40.718032,-122.288295,381,3.434722,2021


In [13]:
# ============================================================
# Cell 10 — Negative sampling (uniform, rejection not clipping)
# ============================================================
np.random.seed(42)

def sample_negative_global(all_detections, bbox_tuple, season, radius_deg=0.3, min_gap_days=14, max_attempts=50):
    west, south, east, north = bbox_tuple
    for _ in range(max_attempts):
        lat = np.random.uniform(south, north)
        lon = np.random.uniform(west, east)

        season_start = pd.Timestamp(f"{season}-08-01")
        season_end = pd.Timestamp(f"{season}-10-31")
        rand_days = np.random.randint(0, (season_end - season_start).days)
        candidate_time = season_start + pd.Timedelta(days=rand_days)

        nearby = all_detections[
            (np.abs(all_detections["latitude"] - lat) < radius_deg) &
            (np.abs(all_detections["longitude"] - lon) < radius_deg) &
            (np.abs((all_detections["acq_datetime"] - candidate_time).dt.total_seconds()) < min_gap_days * 86400)
        ]
        if len(nearby) == 0:
            return {"lat": lat, "lon": lon, "pseudo_time": candidate_time, "season": season}
    return None

negatives = []
failed = 0
for _, row in fire_events_final.iterrows():
    neg = sample_negative_global(firms_clean, BBOX_TUPLE, row["season"])
    if neg:
        negatives.append(neg)
    else:
        failed += 1

negatives_df = pd.DataFrame(negatives)
print(f"Generated {len(negatives_df)} negative samples (target was {len(fire_events_final)}, {failed} failed)")
negatives_df.to_csv("/kaggle/working/raw/negative_samples.csv", index=False)
negatives_df.head()

Generated 452 negative samples (target was 452, 0 failed)


,lat,lon,pseudo_time,season
0,38.247241,-119.246428,2019-10-11,2019
1,39.591951,-123.219907,2019-10-22,2019
2,41.953269,-120.912592,2020-09-27,2020
3,37.023145,-123.674742,2021-08-04,2021
4,36.095798,-122.845531,2021-09-29,2021


In [14]:
# ============================================================
# Cell 11 — Cross-season location leakage check + spatial-cluster train/test split
# ============================================================
cluster_seasons = fire_events_final.groupby('spatial_cluster')['season'].nunique()
leaking_clusters = cluster_seasons[cluster_seasons > 1]
print(f"{len(leaking_clusters)} spatial clusters have events spanning multiple seasons (expected — real terrain reburns)")

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(fire_events_final, groups=fire_events_final['spatial_cluster']))

fire_events_final['split'] = 'train'
fire_events_final.iloc[test_idx, fire_events_final.columns.get_loc('split')] = 'test'

train_clusters = set(fire_events_final[fire_events_final['split']=='train']['spatial_cluster'])
test_clusters = set(fire_events_final[fire_events_final['split']=='test']['spatial_cluster'])
overlap = train_clusters & test_clusters
print(f"Cluster overlap between train/test: {len(overlap)} (must be 0)")
print(fire_events_final['split'].value_counts())

# Assign negatives to splits with matching proportions
train_frac = (fire_events_final['split']=='train').mean()
negatives_df['split'] = np.random.RandomState(42).choice(
    ['train', 'test'], size=len(negatives_df), p=[train_frac, 1-train_frac]
)
print(negatives_df['split'].value_counts())

74 spatial clusters have events spanning multiple seasons (expected — real terrain reburns)
Cluster overlap between train/test: 0 (must be 0)
split
train    366
test      86
Name: count, dtype: int64
split
train    367
test      85
Name: count, dtype: int64


In [15]:
# ============================================================
# Cell 12 — Build unified query point set
# ============================================================
query_points = pd.concat([
    fire_events_final[["lat", "lon", "split"]].assign(
        point_type="positive",
        ref_time=fire_events_final["ignition_time"],
        season=fire_events_final["season"]
    ),
    negatives_df[["lat", "lon", "split"]].assign(
        point_type="negative",
        ref_time=negatives_df["pseudo_time"],
        season=negatives_df["season"]
    )
], ignore_index=True)

print(f"{len(query_points)} total query points")
print(query_points.groupby(["split", "point_type"]).size())
query_points.to_csv("/kaggle/working/raw/query_points.csv", index=False)

904 total query points
split  point_type
test   negative       85
       positive       86
train  negative      367
       positive      366
dtype: int64


In [16]:
# ============================================================
# Cell 13 — Weather pull (Open-Meteo archive, hourly, no key, no station-coverage gaps)
# ============================================================
def pull_openmeteo_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "UTC"
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        hourly = resp.json().get("hourly", {})
        return pd.DataFrame(hourly) if hourly else pd.DataFrame()
    except Exception as e:
        print(f"    FAILED for ({lat},{lon}): {e}")
        return pd.DataFrame()

LOOKBACK_DAYS = 7
weather_frames = []

for idx, row in query_points.iterrows():
    lat, lon = round(row["lat"], 4), round(row["lon"], 4)
    start = (row["ref_time"] - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    end = row["ref_time"].strftime("%Y-%m-%d")

    df = pull_openmeteo_weather(lat, lon, start, end)
    if len(df) > 0:
        df["query_point_idx"] = idx
        df["point_type"] = row["point_type"]
        weather_frames.append(df)
    time.sleep(0.2)

    if idx % 50 == 0:
        print(f"  {idx}/{len(query_points)} processed, {len(weather_frames)} successful pulls so far")

weather_df = pd.concat(weather_frames, ignore_index=True) if weather_frames else pd.DataFrame()
weather_df.to_csv("/kaggle/working/raw/weather_raw.csv", index=False)
print(f"Total weather rows: {len(weather_df)} covering {weather_df['query_point_idx'].nunique() if len(weather_df)>0 else 0}/{len(query_points)} query points")

  0/904 processed, 1 successful pulls so far
  50/904 processed, 51 successful pulls so far
  100/904 processed, 101 successful pulls so far
  150/904 processed, 151 successful pulls so far
  200/904 processed, 201 successful pulls so far
  250/904 processed, 251 successful pulls so far
  300/904 processed, 301 successful pulls so far
    FAILED for (38.0222,-120.7593): HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=15)
  350/904 processed, 350 successful pulls so far
  400/904 processed, 400 successful pulls so far
  450/904 processed, 450 successful pulls so far
  500/904 processed, 500 successful pulls so far
  550/904 processed, 550 successful pulls so far
  600/904 processed, 600 successful pulls so far
    FAILED for (39.88,-120.5341): HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=15)
  650/904 processed, 649 successful pulls so far
  700/904 processed, 699 successful pulls so far
 

In [17]:
# ============================================================
# Cell 14 — OpenAQ functions (corrected datetime_from/datetime_to, rate-limit safe)
# ============================================================
def request_with_retry(url, headers, params=None, max_retries=5, timeout=15):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=timeout)
            if resp.status_code == 429:
                wait = 2 ** attempt
                print(f"    429 hit, backing off {wait}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"    FAILED after {max_retries} attempts: {e}")
                return None
            time.sleep(2 ** attempt)
    return None

def get_locations_near(lat, lon, headers, radius=25000):
    resp = request_with_retry("https://api.openaq.org/v3/locations", headers,
        params={"coordinates": f"{lat},{lon}", "radius": radius, "limit": 5})
    return resp.json().get("results", []) if resp else []

def pull_openaq_measurements(sensor_id, date_from, date_to, headers):
    resp = request_with_retry(f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements", headers,
        params={"datetime_from": date_from, "datetime_to": date_to, "limit": 1000})  # correct param names
    if resp is None:
        return pd.DataFrame()
    results = resp.json().get("results", [])
    rows = [{"sensor_id": sensor_id, "value": r.get("value"),
             "parameter": r.get("parameter", {}).get("name"),
             "datetime": r.get("period", {}).get("datetimeFrom", {}).get("utc")} for r in results]
    return pd.DataFrame(rows)

print("OpenAQ functions defined.")

OpenAQ functions defined.


In [18]:
# ============================================================
# Cell 15 — OpenAQ collection (confirmed rate limit: 60 req/60s -> 1.1s delay, no active-sensor pre-check)
# ============================================================
LOOKBACK_DAYS = 7
openaq_frames = []
location_cache = {}
REQUEST_DELAY = 1.1  # derived from confirmed 60 req/60s OpenAQ limit + margin

for idx, row in query_points.iterrows():
    lat, lon = round(row["lat"], 2), round(row["lon"], 2)
    key = (lat, lon)

    if key not in location_cache:
        location_cache[key] = get_locations_near(lat, lon, OPENAQ_HEADERS, radius=25000)
        time.sleep(REQUEST_DELAY)

    locations = location_cache[key]
    if not locations:
        continue

    loc_id = locations[0]["id"]
    sensors_resp = request_with_retry(f"https://api.openaq.org/v3/locations/{loc_id}/sensors", OPENAQ_HEADERS)
    time.sleep(REQUEST_DELAY)
    if sensors_resp is None:
        continue
    sensors = sensors_resp.json().get("results", [])

    start = (row["ref_time"] - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    end = row["ref_time"].strftime("%Y-%m-%d")

    for sensor in sensors:
        sensor_id = sensor["id"]
        df = pull_openaq_measurements(sensor_id, start, end, OPENAQ_HEADERS)
        if len(df) > 0:
            df["query_point_idx"] = idx
            df["point_type"] = row["point_type"]
            openaq_frames.append(df)
        time.sleep(REQUEST_DELAY)

    if idx % 20 == 0:
        print(f"  {idx}/{len(query_points)} processed, {len(openaq_frames)} successful pulls so far")

openaq_df = pd.concat(openaq_frames, ignore_index=True) if openaq_frames else pd.DataFrame()
if len(openaq_df) > 0:
    openaq_df["datetime"] = pd.to_datetime(openaq_df["datetime"], utc=True)
    print(f"Date range: {openaq_df['datetime'].min()} to {openaq_df['datetime'].max()}")

    # Mandatory verification — this exact check caught a silent bug earlier this session
    expected_ranges = "|".join([f"(({openaq_df['datetime']} >= '{s['start']}') & ({openaq_df['datetime']} <= '{s['end']}'))" for s in SEASONS])
    in_range_mask = pd.Series(False, index=openaq_df.index)
    for s in SEASONS:
        in_range_mask |= (openaq_df["datetime"] >= pd.Timestamp(s["start"]).tz_localize("UTC") - pd.Timedelta(days=10)) & \
                          (openaq_df["datetime"] <= pd.Timestamp(s["end"]).tz_localize("UTC"))
    print(f"Rows outside expected season windows: {(~in_range_mask).sum()} (should be 0 or near 0)")

openaq_df.to_csv("/kaggle/working/raw/openaq_raw.csv", index=False)
print(f"Total: {len(openaq_df)} rows, {openaq_df['query_point_idx'].nunique() if len(openaq_df)>0 else 0}/{len(query_points)} points")

  0/904 processed, 0 successful pulls so far
  20/904 processed, 11 successful pulls so far
  40/904 processed, 26 successful pulls so far
  60/904 processed, 46 successful pulls so far
  80/904 processed, 79 successful pulls so far
  120/904 processed, 109 successful pulls so far
  140/904 processed, 131 successful pulls so far
  160/904 processed, 154 successful pulls so far
  180/904 processed, 161 successful pulls so far
  200/904 processed, 167 successful pulls so far
  220/904 processed, 183 successful pulls so far
  240/904 processed, 188 successful pulls so far
  260/904 processed, 212 successful pulls so far
  280/904 processed, 224 successful pulls so far
    FAILED after 5 attempts: 500 Server Error: Internal Server Error for url: https://api.openaq.org/v3/locations/229509/sensors
  340/904 processed, 258 successful pulls so far
  360/904 processed, 284 successful pulls so far
  380/904 processed, 306 successful pulls so far
  400/904 processed, 312 successful pulls so far
 

In [19]:
out_of_range = openaq_df[~(
    (openaq_df["datetime"] >= "2019-07-20") & (openaq_df["datetime"] <= "2019-11-01") |
    (openaq_df["datetime"] >= "2020-07-20") & (openaq_df["datetime"] <= "2020-11-01") |
    (openaq_df["datetime"] >= "2021-07-20") & (openaq_df["datetime"] <= "2021-11-01") |
    (openaq_df["datetime"] >= "2022-07-20") & (openaq_df["datetime"] <= "2022-11-01") |
    (openaq_df["datetime"] >= "2023-07-20") & (openaq_df["datetime"] <= "2023-11-01") |
    (openaq_df["datetime"] >= "2024-07-20") & (openaq_df["datetime"] <= "2024-11-01")
)]
print(out_of_range[["datetime", "query_point_idx"]])

Empty DataFrame
Columns: [datetime, query_point_idx]
Index: []


In [20]:
covered = openaq_df['query_point_idx'].unique()
query_points['has_openaq'] = query_points.index.isin(covered)
print(query_points.groupby('point_type')['has_openaq'].mean())

point_type
negative    0.123894
positive    0.398230
Name: has_openaq, dtype: float64


In [21]:
# ============================================================
# Cell 1 — Setup and auth
# ============================================================
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("200205")
login(token=HF_TOKEN)
api = HfApi()

REPO_ID = "Jahid05/wildfire-early-warning-sensors-v2"  # your 6-season repo

api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=True, exist_ok=True)
print(f"Repo ready: https://huggingface.co/datasets/{REPO_ID}")

Repo ready: https://huggingface.co/datasets/Jahid05/wildfire-early-warning-sensors-v2


In [22]:
# ============================================================
# Cell 2 — Dataset card, written against confirmed 6-season numbers
# ============================================================
dataset_card = """---
license: cc-by-4.0
tags:
  - wildfire
  - early-warning
  - time-series
  - multimodal
  - sensor-fusion
  - remote-sensing
task_categories:
  - time-series-forecasting
  - tabular-classification
---

# Wildfire Early Warning: Multi-Sensor Fusion Dataset (Northern California, 2019-2024)

## Overview
Multi-sensor dataset for early wildfire ignition prediction research, combining satellite fire detections, historical hourly weather (reanalysis), and ground-level air quality measurements. Built to study accuracy-vs-lead-time tradeoffs and sensor-modality ablation for early-warning systems, across six consecutive fire seasons.

## Region and Period
- Bounding box: -124, 36, -119, 42 (Northern California)
- Seasons: Aug-Oct, 2019 through 2024 (six fire seasons)

## Files

### `fire_events_final.csv` — Positive class (452 events)
Distinct wildfire ignition events, derived from VIIRS satellite hotspot detections (VIIRS_SNPP_SP, Standard Processing archive product) via spatial clustering (DBSCAN, eps=0.05°) then temporal splitting (7-day gap threshold separates distinct ignitions at the same location — real terrain reburns across seasons; 74 of the spatial clusters span multiple seasons). Filtered to events with >=3 detections and >=0.5 days duration to exclude single-pixel false positives and transient heat sources.
- Columns: `event_id`, `spatial_cluster`, `ignition_time`, `last_detection_time`, `lat`, `lon`, `n_detections`, `duration_days`, `season`, `split`
- Events per season: 2019: 93, 2020: 65, 2021: 63, 2022: 59, 2023: 101, 2024: 71

### `negative_samples.csv` — Negative class (452 samples)
Non-fire space-time points, sampled uniformly across the study bbox with rejection (not clipping) against any real detection within 0.3° / 14 days of the candidate point/time.
- Columns: `lat`, `lon`, `pseudo_time`, `season`, `split`

### `query_points.csv` — Unified reference table (904 points)
Union of positive and negative points with a shared `ref_time` and `split` (train/test) column. Row index (`query_point_idx`) is the join key for the sensor files below.
- Split: 452 positive (366 train / 86 test), 452 negative (367 train / 85 test)

### `weather_raw.csv` — Hourly weather (Open-Meteo ERA5 reanalysis)
Hourly temperature, relative humidity, wind speed, and precipitation for the 7 days preceding each query point's `ref_time`. Coverage: 902/904 points (99.8%) — gap-free gridded reanalysis, not station-dependent; the 2 missing points failed on transient read-timeouts.
- **Source**: Open-Meteo Historical Weather API (ERA5/ERA5-Land reanalysis, CC BY 4.0)
- **Note**: reanalysis/model output assimilating station, satellite, and radar observations — not raw station readings.

### `openaq_raw.csv` — Ground-level air quality
Measurements from the nearest OpenAQ regulatory monitoring station within 25km, for the same 7-day pre-event window. Coverage: 236/904 points (26.1%).
- **Source**: OpenAQ v3 API (CC BY 4.0)
- **Known limitation — coverage skew**: OpenAQ coverage correlates with `point_type` (positive-class 39.8% vs. negative-class 12.4%, roughly 3.2x), an artifact of spatial-cluster location reuse across sub-events at the same coordinates, not evidence that fire-prone terrain has better air-quality monitoring.
- **Known limitation — parameter mismatch**: surviving OpenAQ parameters after sparse-column filtering (>50% missing dropped) skew toward ozone (O3) rather than PM2.5, since many matched stations are positioned for ecosystem/visibility regulation (e.g., national park monitors) rather than smoke tracking. PM2.5 is present but was one of the sparser parameters in the matched subset.

## Train/Test Split Methodology
Split by `spatial_cluster` (physical location), not by season. An initial season-based split was found to leak locations across seasons (fire-prone terrain reburns), which would let a model partially memorize locations rather than generalize. The final split uses `GroupShuffleSplit` on `spatial_cluster` with test_size=0.25, verified to have zero cluster overlap between train and test.

## Feature Arms (Sensor-Ablation Design)
Because OpenAQ coverage is sparse and skewed (see above), naive comparison of OpenAQ-inclusive arms against the full weather-only sample confounds sensor-modality effects with sample-size and class-balance differences. Downstream feature-extraction code builds five paired arms (`weather_only`, `weather_matched`, `combined`, `openaq_only`, `missingness_aware`) to isolate this — see repository code / paper methods section for the extraction pipeline and paired-comparison design.

## Known Limitations
- OpenAQ coverage is sparse, non-random, and skewed toward ozone rather than PM2.5 — treat as a genuinely limited modality, not impute-and-ignore.
- `weather_raw.csv` is reanalysis, not observed station data.
- Fire event boundaries use a 7-day gap threshold to separate distinct ignitions at the same location; this is a modeling choice, not ground truth.
- 2 query points have no weather data due to transient API timeouts during collection (out of 904).

## Data Sources and Attribution
- Fire detections: NASA FIRMS, VIIRS S-NPP Standard Processing product (public domain, please cite: https://earthdata.nasa.gov/firms)
- Weather: Open-Meteo (CC BY 4.0): https://open-meteo.com
- Air quality: OpenAQ (CC BY 4.0): https://openaq.org

## Usage
```python
from huggingface_hub import hf_hub_download
import pandas as pd

REPO_ID = "REPO_ID"
query_points = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="query_points.csv", repo_type="dataset"))
weather_df = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="weather_raw.csv", repo_type="dataset"))
openaq_df = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="openaq_raw.csv", repo_type="dataset"))
fire_events = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="fire_events_final.csv", repo_type="dataset"))
negatives = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="negative_samples.csv", repo_type="dataset"))
```
"""

with open("/kaggle/working/README.md", "w") as f:
    f.write(dataset_card.replace('"REPO_ID"', f'"{REPO_ID}"'))

print("Dataset card written.")

Dataset card written.


In [23]:
# ============================================================
# Cell 3 — Upload everything
# ============================================================
files_to_upload = [
    "/kaggle/working/README.md",
    "/kaggle/working/raw/fire_events_final.csv",
    "/kaggle/working/raw/negative_samples.csv",
    "/kaggle/working/raw/query_points.csv",
    "/kaggle/working/raw/weather_raw.csv",
    "/kaggle/working/raw/openaq_raw.csv",
]

for filepath in files_to_upload:
    filename = filepath.split("/")[-1]
    api.upload_file(
        path_or_fileobj=filepath,
        path_in_repo=filename,
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print(f"Uploaded: {filename}")

print(f"\nDataset live at: https://huggingface.co/datasets/{REPO_ID}")

Uploaded: README.md
Uploaded: fire_events_final.csv
Uploaded: negative_samples.csv
Uploaded: query_points.csv
Uploaded: weather_raw.csv
Uploaded: openaq_raw.csv

Dataset live at: https://huggingface.co/datasets/Jahid05/wildfire-early-warning-sensors-v2
